# Comparison of urban indicators in Tampere and Helsinki, Finland

The following Jupyter notebook calculates some urban indicators for Tampere and Helsinki.  

In [1]:
import pathlib
import os
import logging
if not os.path.exists("logs"):
    os.makedirs("logs")
NOTEBOOK_PATH = pathlib.Path().resolve()
LOG_DIRECTORY = NOTEBOOK_PATH / 'logs'
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.DEBUG, 
                    format='%(asctime)s %(levelname)-8s %(message)s', 
                    datefmt='%a, %d %b %Y %H:%M:%S', 
                    filename=LOG_DIRECTORY / 'FAlog.log', filemode='w')
#logging.disable(logging.NOTSET)
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"

In [65]:
# Most necessary imports
import json

import geopandas as gpd
import pandas as pd
# import requests
# from datetime import datetime
import matplotlib.pyplot as plt
# import contextily as cx
import networkx as nx
import osmnx as ox
#import street_network_metrics as net
# Set counter for table caption numbers
global tcounter
tcounter = 0

os.environ['PROJ_LIB'] = '/path/to/env/share/proj'
#os.environ['GDAL_DATA'] = '/path/to/env/share/'
pd.set_option("display.max_rows", None)

In [15]:
CITIES = CITY1, CITY2 = ('Helsinki, Finland', 'Tampere, Finland')
logger.info('Notebook starts. Cities selected: {c1}; {c2}.'.format(c1=CITY1, c2=CITY2))
logger.info('Calculating network statistics')

In [16]:
def get_network(osm_city, network_type='bike'):
    assert type(osm_city) == str, 'You should provide a string'
    assert osm_city is not None, 'You should provide a city name'
    logger.info('Start downloading network for {city}, network_type={t}'.format(city=osm_city, t=network_type))
    graph = ox.graph_from_place(osm_city, network_type=network_type)
    assert graph is not None, 'No graphs returned by the query'
    logger.info('Network downloaded for {city}. Object: {g}'.format(city=osm_city, t=type(graph), g=graph))
    return osm_city, graph

In [5]:
# nets_t = get_networks([])
# for G in nets:
#     print(f)
nets = { c: {'graph': g} for c, g in [get_network(city) for city in CITIES]}

In [6]:
print(type(nets))
print(len(nets.keys()))
for key in nets:
    print(nets[key])
    #ox.plot_graph(G)

<class 'dict'>
2
{'graph': <networkx.classes.multidigraph.MultiDiGraph object at 0x7f3478f4ab00>}
{'graph': <networkx.classes.multidigraph.MultiDiGraph object at 0x7f3488ac5bd0>}


In [7]:
def graphs_to_gdfs(graph_objs):
    assert len(graph_objs.keys()) > 0, 'No graphs to convert'
    for key in graph_objs:
        logger.info('Reprojecting graph: {city}'.format(city=key))
        graph = ox.project_graph(graph_objs[key]['graph'])
        logger.info('Start converting to a GeoDataFrame: {city}'.format(city=key))
        nodes, edges = ox.graph_to_gdfs(graph)
        graph_objs[key]['edges'] = edges
        graph_objs[key]['nodes'] = nodes
        logger.info('Conversion DONE: {city}'.format(city=key))
    return graph_objs

In [8]:
gdf_dict = graphs_to_gdfs(nets)


In [58]:
def gdf_value_counts(cityname, gdf, col_name, type_title, explode=False):
    
    def list_to_str(ls):
        if all([len(elem) > 1 and type(elem) == str for elem in ls]):
            return ", ".join(ls)
        else:
            return ls
    
    logger.info('Extracting value counts: {city}, exploded={exp}.'.format(city=cityname, exp=explode))
    renamed_col = 'Count ({city})'.format(city=cityname)
    if explode:
        logger.info('Exploding GeoDataFrame: {city}'.format(city=cityname))
        gdf = gdf.explode(col_name)
        logger.info('Exploding DONE: {city}'.format(city=cityname))
    counts = pd.DataFrame(gdf[col_name].value_counts().reset_index())
    if not explode:
        counts['index'] = counts['index'].apply(list_to_str)
    counts[col_name] = counts[col_name].astype(pd.Int64Dtype())
    logger.info('Renaming columns: {col}'.format(col=col_name))
    counts = counts.rename({col_name: renamed_col, 'index': type_title}, axis=1)
    return renamed_col, counts

In [61]:
def join_highway_type_counts(gdf_dict, join_col):
    logger.info('Start joining highway type counts on "{key}"'.format(key=join_col))
    for city in gdf_dict:
        gdf = gdf_dict[city]['edges']
        count_col_exp, counts_exp = gdf_value_counts(city, gdf, 'highway', join_col, explode=True)
        count_col, counts = gdf_value_counts(city, gdf, 'highway', join_col, explode=False)
        gdf_dict[city]['exp_counts'] = counts_exp
        gdf_dict[city]['counts'] = counts
    left_exp, right_exp = gdf_dict[CITY1]['exp_counts'], gdf_dict[CITY2]['exp_counts']
    joined_counts_exp = left_exp.merge(right=right_exp, left_on=join_col, right_on=join_col, how='outer')
    left, right = gdf_dict[CITY1]['counts'], gdf_dict[CITY2]['counts']
    joined_counts = left.merge(right=right, left_on=join_col, right_on=join_col, how='outer')
    logger.info('Type counts joined on "{key}"'.format(key=join_col))
    return joined_counts_exp, joined_counts

In [62]:
total_counts_exp, total_counts = join_highway_type_counts(gdf_dict, 'Roadlink types')
total_counts_exp = total_counts_exp.set_index('Roadlink types')
total_counts = total_counts.set_index('Roadlink types')
print(total_counts_exp.info())
print(total_counts.info())

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, service to road
Data columns (total 2 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Count (Helsinki, Finland)  17 non-null     Int64
 1   Count (Tampere, Finland)   17 non-null     Int64
dtypes: Int64(2)
memory usage: 468.0+ bytes
None
<class 'pandas.core.frame.DataFrame'>
Index: 104 entries, service to path, track, residential, service
Data columns (total 2 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Count (Helsinki, Finland)  89 non-null     Int64
 1   Count (Tampere, Finland)   68 non-null     Int64
dtypes: Int64(2)
memory usage: 2.6+ KB
None


In [63]:
def captioned_table(df, f_str, format_keys):
    
    def numbered_table_caption(f_str):
        global tcounter
        tcounter += 1
        return 'Table {tc}. '.format(tc=tcounter) + f_str 
    
    styled = df.style.set_table_attributes("style='display:inline'")
    styled = styled.set_caption(numbered_table_caption(f_str).format(**format_keys))
    return styled

In [70]:
total_counts_styled_exp = captioned_table(total_counts_exp, 
                                      ('Number of roadlinks per roadlink type in {city1} and {city2}. ' 
                                       'Some roadlinks belong to several categories.'), dict(city1=CITY1, city2=CITY2))
total_counts_styled_exp

,"Count (Helsinki, Finland)","Count (Tampere, Finland)"
Roadlink types,,
service,51085,23508
cycleway,46939,22910
residential,31651,18239
path,30436,26578
tertiary,7840,4195
secondary,5747,1790
unclassified,5606,3018
track,2310,4163
pedestrian,1714,168


In [71]:
styled_value_counts = captioned_table(total_counts, ('Roadlink type sets for {city1} and {city2}. ' 
                                       'Some roadlinks belong to several categories but the sets are exclusive.'), 
                                      dict(city1=CITY1, city2=CITY2))

In [72]:
styled_value_counts

,"Count (Helsinki, Finland)","Count (Tampere, Finland)"
Roadlink types,,
service,49466,22586
cycleway,45615,22066
residential,30149,17110
path,29704,25866
tertiary,7823,4186
secondary,5735,1787
unclassified,5345,2925
track,2052,3821
pedestrian,1502,156
